In [0]:
#Read CSV Files into Spark DataFrames
df_airports = spark.read.csv(
    "/Volumes/vita_catalog/dataengineering/data/airports.csv",
    header=True,
    inferSchema=True
)

df_runways = spark.read.csv(
    "/Volumes/vita_catalog/dataengineering/data/runways.csv",
    header=True,
    inferSchema=True
)

df_airports.printSchema()
df_runways.printSchema()

df_airports.show(5)
df_runways.show(5)

root
 |-- id: integer (nullable = true)
 |-- ident: string (nullable = true)
 |-- type: string (nullable = true)
 |-- name: string (nullable = true)
 |-- latitude_deg: double (nullable = true)
 |-- longitude_deg: double (nullable = true)
 |-- elevation_ft: integer (nullable = true)
 |-- continent: string (nullable = true)
 |-- iso_country: string (nullable = true)
 |-- iso_region: string (nullable = true)
 |-- municipality: string (nullable = true)
 |-- scheduled_service: string (nullable = true)
 |-- icao_code: string (nullable = true)
 |-- iata_code: string (nullable = true)
 |-- gps_code: string (nullable = true)
 |-- local_code: string (nullable = true)
 |-- home_link: string (nullable = true)
 |-- wikipedia_link: string (nullable = true)
 |-- keywords: string (nullable = true)

root
 |-- id: integer (nullable = true)
 |-- airport_ref: integer (nullable = true)
 |-- airport_ident: string (nullable = true)
 |-- length_ft: integer (nullable = true)
 |-- width_ft: integer (nullable = 

In [0]:
#Inner JOin
df_combined = df_airports.join(
    df_runways,
    df_airports.ident == df_runways.airport_ident,
    "inner"
)

df_combined.show(5)

+------+-----+-------------+--------------------+------------+-------------+------------+---------+-----------+----------+------------+-----------------+---------+---------+--------+----------+--------------------+--------------+--------+------+-----------+-------------+---------+--------+-------+-------+------+--------+---------------+----------------+---------------+---------------+-------------------------+--------+---------------+----------------+---------------+---------------+-------------------------+
|    id|ident|         type|                name|latitude_deg|longitude_deg|elevation_ft|continent|iso_country|iso_region|municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|           home_link|wikipedia_link|keywords|    id|airport_ref|airport_ident|length_ft|width_ft|surface|lighted|closed|le_ident|le_latitude_deg|le_longitude_deg|le_elevation_ft|le_heading_degT|le_displaced_threshold_ft|he_ident|he_latitude_deg|he_longitude_deg|he_elevation_ft|he_heading_degT

In [0]:
#Add runway category column
from pyspark.sql.functions import when, col

df_combined = df_combined.withColumn(
    "runway_category",
    when(col("length_ft") >= 10000, "Long")
    .when((col("length_ft") >= 5000) & (col("length_ft") < 10000), "Medium")
    .otherwise("Short")
)

df_combined.select(
    "name",
    "length_ft",
    "runway_category"
).show(10)

+--------------------+---------+---------------+
|                name|length_ft|runway_category|
+--------------------+---------+---------------+
|   Total RF Heliport|       80|          Short|
|        Lowell Field|     2500|          Short|
|        Epps Airpark|     2100|          Short|
|Katmai Lodge Airport|     4517|          Short|
|      Fulton Airport|     1450|          Short|
|      Cordes Airport|     1700|          Short|
|Goldstone (GTS) A...|     6000|         Medium|
| Grass Patch Airport|     3200|          Short|
|   River Oak Airport|     4700|          Short|
|    Lt World Airport|     2600|          Short|
+--------------------+---------+---------------+
only showing top 10 rows


In [0]:
#Temporary View
df_combined.createOrReplaceTempView("airports_view")

In [0]:
%sql
SELECT
    iso_country,
    COUNT(*) AS runway_count
FROM airports_view
GROUP BY iso_country
ORDER BY runway_count DESC
LIMIT 10

iso_country,runway_count
US,26450
BR,5825
AU,2103
CA,1578
AR,761
FR,659
GB,600
DE,592
RU,489
ID,414


In [0]:
%sql
SELECT
    continent,
    AVG(length_ft) AS avg_runway_length
FROM airports_view
WHERE length_ft IS NOT NULL
GROUP BY continent
ORDER BY avg_runway_length DESC

continent,avg_runway_length
AN,7768.857142857143
AS,6925.6469996647675
AF,6787.526448362721
EU,4103.151335311572
OC,3558.0401785714284
SA,2857.0891141239963
NA,2599.724218941758


In [0]:
%sql
SELECT
    name,
    municipality,
    COUNT(*) AS lighted_runways
FROM airports_view
WHERE iso_country = 'IN'
AND lighted = 1
GROUP BY name, municipality
ORDER BY lighted_runways DESC

name,municipality,lighted_runways
Indira Gandhi International Airport,New Delhi,3
Navi Mumbai International Airport,Navi Mumbai,2
Kempegowda International Airport Bengaluru,Bengaluru,2
Mangaluru International Airport,Mangaluru,2
Chennai International Airport,Chennai,2
Ambala Air Force Station,Ambala,2
Bidar Airport / Bidar Air Force Station,Bidar,2
Agra Airport / Agra Air Force Station,Agra,2
Rajiv Gandhi International Airport,Hyderabad,2
Raja Bhoj International Airport,Bhopal,2


In [0]:
%sql
SELECT
    name,
    iso_country,
    MAX(length_ft) AS max_runway_length
FROM airports_view
GROUP BY name, iso_country
ORDER BY max_runway_length DESC
LIMIT 5

name,iso_country,max_runway_length
Gunflint Seaplane Base,US,30000
Libby Camps Seaplane Base,US,26000
Brookville Reservoir Seaplane Base,US,25000
Long Lake Seaplane Base,US,25000
Conchas Lake Seaplane Base,US,21120


In [0]:
df_combined = df_combined.drop(df_runways.id)

In [0]:
df_combined.write.mode("overwrite").parquet(
    "/Volumes/vita_catalog/dataengineering/data/airports_runways_parquet"
)

In [0]:
df_parquet = spark.read.parquet(
    "/Volumes/vita_catalog/dataengineering/data/airports_runways_parquet"
)

df_parquet.show(10)

+------+-------+-------------+--------------------+------------+-------------+------------+---------+-----------+----------+-------------+-----------------+---------+---------+--------+----------+---------+--------------------+--------------------+-----------+-------------+---------+--------+--------+-------+------+--------+---------------+----------------+---------------+---------------+-------------------------+--------+---------------+----------------+---------------+---------------+-------------------------+---------------+
|    id|  ident|         type|                name|latitude_deg|longitude_deg|elevation_ft|continent|iso_country|iso_region| municipality|scheduled_service|icao_code|iata_code|gps_code|local_code|home_link|      wikipedia_link|            keywords|airport_ref|airport_ident|length_ft|width_ft| surface|lighted|closed|le_ident|le_latitude_deg|le_longitude_deg|le_elevation_ft|le_heading_degT|le_displaced_threshold_ft|he_ident|he_latitude_deg|he_longitude_deg|he_elev